**Importing libraries**

In [1]:
import pynarrative as pn
import altair as alt
import pandas as pd

#Importing template
from pynarrative.templates.mytemplates.darkBlueTemplate import darkBlueTemplate

**Importing fonts**

This cell contains the font URLs from Google Fonts, which are essential for displaying the correct font in the template.

In [2]:
%%html
<style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;700&display=swap');
    @import url('https://fonts.googleapis.com/css2?family=Lato:wght@400;700&display=swap');
    @import url('https://fonts.googleapis.com/css2?family=Lora:wght@400;700&display=swap');
    @import url('https://fonts.googleapis.com/css2?family=Merriweather:wght@400;700&display=swap');
    @import url('https://fonts.googleapis.com/css2?family=Montserrat:wght@400;700&display=swap');
    @import url('https://fonts.googleapis.com/css2?family=Noto+Sans:wght@400;700&display=swap');
    @import url('https://fonts.googleapis.com/css2?family=Nunito:wght@400;700&display=swap');
    @import url('https://fonts.googleapis.com/css2?family=Open+Sans:wght@400;700&display=swap');
    @import url('https://fonts.googleapis.com/css2?family=Oswald:wght@400;700&display=swap');
    @import url('https://fonts.googleapis.com/css2?family=Playfair+Display:wght@400;700&display=swap');
    @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;700&display=swap');
    @import url('https://fonts.googleapis.com/css2?family=Raleway:wght@400;700&display=swap');
    @import url('https://fonts.googleapis.com/css2?family=Roboto+Condensed:wght@400;700&display=swap');
    @import url('https://fonts.googleapis.com/css2?family=Roboto:wght@400;700&display=swap');
    @import url('https://fonts.googleapis.com/css2?family=Ubuntu:wght@400;700&display=swap');
    @import url('https://fonts.googleapis.com/css2?family=Bricolage+Grotesque:opsz,wght@12..96,400;700&display=swap');
    @import url('https://fonts.googleapis.com/css2?family=Cinzel+Decorative:wght@400;700&display=swap');
    @import url('https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@400;700&display=swap');
    @import url('https://fonts.googleapis.com/css2?family=Pacifico&display=swap');
    @import url('https://fonts.googleapis.com/css2?family=Syne:wght@400;700&display=swap');
</style>

**Creating dataframes**

Data source: *Italian Ministero della Cultura via statista.com* and *Société d'exploitation de la tour Eiffel via statista.com*

In [3]:
year = [y  for y in range(2012, 2025)]

colosseum = pd.DataFrame({
    "year" : year,
    "visitors" : [5.201, 5.625, 6.182, 6.551, 6.409, 7.036, 7.650, 7.618, 1.086, 1.689, 9.812, 12.298, 14.733]
})

year = [y  for y in range(2011, 2026)]

tour_eiffel = pd.DataFrame({
    "year" : year,
    "visitors" : [7.08, 6.27, 6.74, 7.10, 6.92, 5.84, 6.23, 6.07, 6.14, 1.16, 2.06, 5.85, 6.32, 6.30, 6.74]
})

**Merging dataframes**

In [4]:
colosseum_vs_tour_eiffel = pd.merge(
    left = colosseum, right = tour_eiffel, how = "left", on = "year"
)

colosseum_vs_tour_eiffel.rename(columns=({
    "visitors_x" : "colosseum",
    "visitors_y" : "tour_eiffel",
}),
inplace = True
)

To plot data with series we must **melt** the dataset, meaning **reshaping DataFrame from wide format to long (tidy) format**

In [5]:
colosseum_vs_tour_eiffel_long = colosseum_vs_tour_eiffel.melt(
    id_vars = "year",                          # Column that remains fixed as the key identifier
    value_vars = ["colosseum", "tour_eiffel"],  # Columns to transform into row values
    var_name = "monument",                     # Name of the new categorical column
    value_name = "visitors"                    # Name of the new column containing numeric values
)

Data story with **bar chart**

In [6]:
story = (
    pn.Story(
    #Builing the Story class object
        data = colosseum_vs_tour_eiffel_long, #Using the long (melted) dataframe!
        width = 900,
        height = 300,
        template = darkBlueTemplate #Using darkBlueTemplate
    )

    #Method invocation
    #Bar chart
    .mark_bar(
        cornerRadiusEnd = 10,
        size = 20
    )

    #Data encoding
    .encode(
        x = alt.X(
            "year:O",
            title = "Year", 
            axis = alt.Axis(
                grid = True,
                labelAngle = -45,
            ),
            scale = alt.Scale(paddingInner = 0.5) #Adjust spacing between different year groups
        ),
        y = alt.Y(
            "visitors:Q",
            title = "Number of visitors (in millions)",
            axis = alt.Axis(
                grid = True
            ),
            scale = alt.Scale(domain=[0, 16]) #Set fixed range for Y axis from 0 to 16 million
        ),
        color = alt.Color("monument:N", title = "Colosseum vs Tour Eiffel"),
        xOffset = alt.XOffset("monument:N", scale = alt.Scale(paddingInner = 0.5)) #Adjust inner spacing between individual bars of the same year
    )
    


    #Data source
    .add_source(
        text = "Source: Ministero della Cultura and Société d'exploitation de la tour Eiffel via statista.com",
        position = "top",
        align = "right"
    )
    
    #Title and subtitle
    .add_title(
        title = "Colosseum Archaeological Park vs Eiffel Tower number of visitors",
        subtitle = "from 2012 to 2024",
        align = "center"
    )

    #Context area (on the left)
    .add_context(
        text = "From 2012 to 2019, visitor figures for both monuments remained closely aligned, with the Eiffel Tower maintaining a slight lead until 2016, when the Colosseum Archaeological Park began to pull ahead. Following a sharp decline during the 2020–2021 pandemic restrictions, the two landmarks experienced starkly different post-pandemic recoveries. While attendance at the Eiffel Tower stabilized near pre-pandemic levels (around 6.3 million in 2024), visitor numbers at the Colosseum surged dramatically, surging past 14.7 million in 2024 and creating a significant divergence between the two iconic sites.",
        position = "left",
    )

    .render()
)

story


alt.VConcatChart(...)

Data story with **line chart**

In [ ]:
story = (
    pn.Story(
    #Builing the Story class object
        data = colosseum_vs_tour_eiffel_long, #Using the long (melted) dataframe!
        width = 600,
        height = 300,
        template = darkBlueTemplate #Using darkBlueTemplate
    )

    #Method invocation
    #Line chart
    .mark_line(
        point = True,
        interpolate = "cardinal"
    )

    #Data encoding
    .encode(
        x = alt.X(
            "year:O",
            title = "Year", 
            axis = alt.Axis(
                grid = True,
                labelAngle = -45,
            ),
        ),
        y = alt.Y(
            "visitors:Q",
            title = "Number of visitors (in millions)",
            axis = alt.Axis(
                grid = True
            ),
            scale = alt.Scale(domain=[0, 16])
        ),
        color = alt.Color("monument:N", title = "Colosseum vs Tour Eiffel"),
    )
    


    #Data source
    .add_source(
        text = "Source: Ministero della Cultura and Société d'exploitation de la tour Eiffel via statista.com",
        position = "top",
        align = "right"
    )
    
    #Title and subtitle
    .add_title(
        title = "Colosseum Archaeological Park vs Eiffel Tower number of visitors",
        subtitle = "from 2012 to 2024",
        align = "center"
    )

    #Context area (on the right)
    .add_context(
        text = "From 2012 to 2019, visitor figures for both monuments remained closely aligned, with the Eiffel Tower maintaining a slight lead until 2016, when the Colosseum Archaeological Park began to pull ahead. Following a sharp decline during the 2020–2021 pandemic restrictions, the two landmarks experienced starkly different post-pandemic recoveries. While attendance at the Eiffel Tower stabilized near pre-pandemic levels (around 6.3 million in 2024), visitor numbers at the Colosseum surged dramatically, surging past 14.7 million in 2024 and creating a significant divergence between the two iconic sites.",
        position = "right",
    )

    .render()
)

story


alt.VConcatChart(...)